In [1]:
import logfire

from src.agents.agentic.query_expander import QueryExpander
from src.common.llm.groq import GroqClient
from src.common.services.hybrid_search import HybridSearch
from src.common.services.qdrant import QdrantStorageService
from src.common.services.reranker import Reranker
from src.common.utils.config import config
from src.ingestion.embedding import EmbeddingService

/home/mano/Manoj/Learning/k_academy/advanced_rag/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
logfire.configure(service_name="Reranking")

Logfire project URL: https://logfire-us.pydantic.dev/manojee/studious

In [3]:
query = "What is transformer in LLM"

In [4]:
groq_client = GroqClient(timeout_seconds=30, max_retries=2)

In [5]:
query_expander = QueryExpander(groq_client)

In [6]:
expanded_queries = await query_expander.expand(query)

19:17:01.338 llm.complete
19:17:01.906 Query expanded to 3 variants
19:17:01.908   - What is transformer in LLM
19:17:01.908   - LLM transformer definition
19:17:01.909   - Transformer role in LLM


In [7]:
expanded_queries

['What is transformer in LLM',
 'LLM transformer definition',
 'Transformer role in LLM']

In [8]:
embedding_service = EmbeddingService(
    model_name=config.EMBEDDING_MODEL_NAME,
    dimensions=config.EMBEDDING_DIMENSIONS,
    batch_size=config.EMBEDDING_BATCH_SIZE,
)

In [9]:
storage_service = QdrantStorageService(
    url=config.QDRANT_CLUSTER_ENDPOINT,
    vector_size=config.VECTOR_SIZE,
    collection_name=config.QDRANT_COLLECTION_NAME,
)

In [10]:
hybrid_search = HybridSearch(storage_service=storage_service, embedding_service=embedding_service)

In [11]:
hybrid_result = await hybrid_search.search(queries=expanded_queries)

19:17:09.180 hybrid_search_start
19:17:09.184 single_search
19:17:09.185 single_search
19:17:09.187 single_search
19:17:12.805 hybrid_search_result
19:17:12.806 hybrid_search_complete


In [12]:
hybrid_result

[{'text': 'reated in Master PDF Edito Table 4: The Transformer generalizes well to English constituency parsing (Results are on Section 23 of WSJ) increased the maximum output length to input length + 300 . We used a beam size of 21 and α = 0 \ue004 3 for both WSJ only and the semi-supervised setting. Our results in Table 4 show that despite the lack of task-specific tuning our model performs surprisingly well, yielding better results than all previously reported models with the exception of the Recurrent Neural Network Grammar [8]. In contrast to RNN sequence-to-sequence models [37], the Transformer outperforms the BerkeleyParser [29] even when training only on the WSJ training set of 40K sentences. 7 Conclusion In this work, we presented the Transformer, the first sequence transduction model based entirely on attention, replacing the recurrent layers most commonly used in encoder-decoder architectures with multi-headed self-attention. For translation tasks, the Transformer can be tra

In [ ]:
reranker = Reranker()
rerank_result = await reranker.rerank(query=query, candidates=hybrid_result)

In [ ]:
rerank_result